In [ ]:
import pandas as pd 
import numpy as np 
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime
import os
import geopandas as gpd

In [ ]:
%run ./methods/methods.py


### 1. General statistics on identified biais  

In [ ]:
directory_path = './data/outputs/'
for file in os.listdir(directory_path):
    if 'wstreetInfos' in file:
        filepath = os.path.join(directory_path, file)


In [ ]:
df = pd.read_csv(filepath,sep=";").drop('Unnamed: 0',axis=1)

In [ ]:
df_biaises = pd.read_csv('./data/biaises_identified.csv',delimiter="|",names=['index','biais'], index_col='index')
df_biaises = df_biaises.drop(index =[np.nan],axis=0)


In [ ]:
df_find = df.copy()


biaises = pd.Series(df_biaises['biais'].tolist())
df_positions = find_pos_elem(df_find, 'adr_init',biaises,'biais','contains_biais','pos_biais',drop_col_elem=False)




In [ ]:
df_w_street = df_positions[df_positions['contains_street_type']==True]
df_wout_street = df_positions[df_positions['contains_street_type']==False]
df_w_biais = df_positions[df_positions['contains_biais']==True]
df_wout_biais = df_positions[df_positions['contains_biais']==False]

print(f"Nombre total d'adresse :{len(df_positions)}")

print(f"Nombre d'adresse contenant un type de voies parmis ceux identifiés : {len(df_w_street)}, soit {np.round(len(df_w_street)/len(df_positions)*100,3)} %")
print(f"Nombre d'adresse ne contenant pas un type de voies parmis ceux identifiés : {len(df_wout_street)}, soit {np.round(len(df_wout_street)/len(df_positions)*100,3)} %")
print(f"Nombre d'adresse contenant un type de biais parmis les 10 identifiés : {len(df_w_biais)}, soit {np.round(len(df_w_biais)/len(df_positions)*100,3)} %")
print(f"Nombre d'adresse ne contenant pas un type de biais parmis les 10 identifiés : {len(df_wout_biais)}, soit {np.round(len(df_wout_biais)/len(df_positions)*100,3)} %")
print("\n")
print(f"Nombre d'adresse contenant un type de voies et un type de biais parmis ceux identifiés : {len(df_positions[df_positions['contains_street_type']&df_positions['contains_biais']])}, \n\t soit {np.round(len(df_positions[df_positions['contains_street_type']&df_positions['contains_biais']])/len(df_positions)*100,3)} % du total des adresses ")

print(f"Nombre d'adresse contenant pas un type de voies ni un type de biais parmis ceux identifiés : {len(df_positions[(df_positions['contains_street_type']==False)&(df_positions['contains_biais']==False)])}, \n\t soit {np.round(len(df_positions[(df_positions['contains_street_type']==False)&(df_positions['contains_biais']==False)])/len(df_wout_street)*100,3)} % du total des adresses sans voie")

print(f"Nombre d'adresse contenant un type de voies et sans biais parmis ceux identifiés : {len(df_positions[df_positions['contains_street_type']&(df_positions['contains_biais']==False)])}, \n\t soit {np.round(len(df_positions[df_positions['contains_street_type']&(df_positions['contains_biais']==False)])/len(df_w_street)*100,3)} % du total des adresses avec une voie")
print(f"Nombre d'adresse contenant un type de biais et sans voies parmis ceux identifiés : {len(df_positions[(df_positions['contains_street_type']==False)&df_positions['contains_biais']])}, \n\t soit {np.round(len(df_positions[(df_positions['contains_street_type']==False)&df_positions['contains_biais']])/len(df_wout_street)*100,3)} % du total des adresses sans voie")


In [ ]:
df_bruite_w_street = df_positions[df_positions['contains_street_type']&df_positions['contains_biais']]
df_bruite_w_street['id'] = df_bruite_w_street.reset_index().index

freq_couverture(df_bruite_w_street,biaises.tolist(),'adr_init').sort_values(by="cumsum",ascending=False)[['cumsum','freq_biais_cum']]

### 2. Impact of biais on geocoded biaised reference 

#### 2.1. Load biased geocoded reference and socio economic data 

In [ ]:
path = "./data/reference/ref_biaised_geocoded/"

# data_all = {}
   
list_files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
i=0
for file in list_files :    
    
    df = pd.read_csv(os.path.join(path,file),sep=";",dtype=str)
    if i==0 : 
        data_all_biaises = df

    else : 
        data_all_biaises = pd.concat([data_all_biaises,df],axis=0)
    i+=1 


##### Download file CONTOURS-IRIS.shp
  
at : https://geoservices.ign.fr/contoursiris  
store it : ./data/socioeco/IRIS/

In [ ]:
df_revenus = pd.read_csv("./data/socioeco/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_iris = gpd.read_file('./data/socioeco/IRIS/CONTOURS-IRIS.shp').set_crs(2154)

#### 2.2. Data Cleaning and pre treatment 

In [ ]:
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan

df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)

df_revenus = df_revenus.rename({"IRIS":"CODE_IRIS"},axis=1).astype(str)
df_revenus = df_revenus.dropna(subset='CODE_IRIS')
df_revenus = df_revenus.drop_duplicates(subset='CODE_IRIS')

df = data_all_biaises.copy()

#### 2.3. Join the reference and geocoded coordinates to socio economic dataset  

In [ ]:
## reference coordinates : x_L93_ref, y_L93_ref
gdf_init = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x_L93_ref, df.y_L93_ref)).set_crs(epsg=2154)
df_join_iris = gpd.sjoin(gdf_init, df_iris[['CODE_IRIS','geometry']], how="left", predicate='within')
df_join_iris.drop('index_right', axis=1, inplace=True)
df_init = df_join_iris.rename({'CODE_IRIS':'CODE_IRIS_init'},axis=1)

## geocoded coordinates : longitude, latitude
gdf_geo = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude)).set_crs(epsg=4326)).to_crs(epsg=2154)
df_join_iris = gpd.sjoin(gdf_geo, df_iris[['CODE_IRIS','geometry']], how="left", predicate='within')
df_join_iris.drop('index_right', axis=1, inplace=True)
df_geo = df_join_iris.rename({'CODE_IRIS':'CODE_IRIS_geocoded'},axis=1)


df_fin = pd.concat([df_init,df_geo['CODE_IRIS_geocoded']],axis=1)


In [ ]:
## Join to get median income associated to initial IRIS code 
df_fin = df_fin.dropna(subset="CODE_IRIS_geocoded")
df_rev_geoc = df_fin.merge(df_revenus,how="left",left_on="CODE_IRIS_geocoded",right_on="CODE_IRIS")
df_rev_geoc = df_rev_geoc.rename({'DISP_MED20':'DISP_MED20_geocoded'},axis=1)
df_rev_geoc = df_rev_geoc.drop('CODE_IRIS',axis=1)

## Join to get median income associated to geocoded IRIS code 
df_rev_init = df_rev_geoc.dropna(subset="CODE_IRIS_init")
df_rev_init = df_rev_init.merge(df_revenus,how="left",left_on="CODE_IRIS_init",right_on="CODE_IRIS")
df_rev_init = df_rev_init.rename({'DISP_MED20':'DISP_MED20_init'},axis=1)

df_rev = df_rev_init.copy()

df_rev["DISP_MED20_geocoded"] = df_rev["DISP_MED20_geocoded"].astype(float)
df_rev["DISP_MED20_init"] = df_rev["DISP_MED20_init"].astype(float)

df_rev["y_WGS84_ref"] = df_rev["y_WGS84_ref"].astype(float)
df_rev["x_WGS84_ref"] = df_rev["x_WGS84_ref"].astype(float)

df_rev["latitude"] = df_rev["latitude"].astype(float)
df_rev["longitude"] = df_rev["longitude"].astype(float)

## Distance between the two coordinates 
df_rev["distance_km"] = calculer_distance_euclidienne(df_rev['y_WGS84_ref'].values, df_rev['x_WGS84_ref'].values, df_rev['latitude'].values, df_rev['longitude'].values)

df_rev["distance_m"] = df_rev['distance_km']*1000

## Difference of median income between the two IRIS  
df_rev["diff_revenu"] =pd.to_numeric(df_rev["DISP_MED20_init"]) - pd.to_numeric(df_rev["DISP_MED20_geocoded"])

In [ ]:
if not os.path.exists('./figures/'):
    os.mkdir('./figures/')

#### Plot distribution difference in both : 

1- Median Income

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

CHU_name = "HEGP"


### diff_revenu
plt.figure(figsize=(6, 6))

ax = sns.boxplot(
    x="label", y="diff_revenu", data=df_rev,
    whis=1.5, showcaps=True, showbox=True, showfliers=False
)

sns.stripplot(
    x="label", y="diff_revenu", data=df_rev,
    jitter=True, color="black", alpha=0.3, size=2, ax=ax
)

# Color the boxes red
for i, artist in enumerate(ax.patches):
    # Each box is a Rectangle object
    artist.set_edgecolor('red')
    artist.set_linewidth(2)
    artist.set_facecolor('none')  # Optional

plt.xticks(rotation=45)
plt.title(f"Boxplot of difference in income per label for {CHU_name} adresses")

plt.tight_layout()
plt.savefig(f'./figures/boxplot_diff_revenu_{CHU_name}.png')
plt.show()


2- Distance from CHU

In [ ]:


plt.figure(figsize=(6, 6))

ax = sns.boxplot(
    x="label", y="distance_km", data=df_rev,
    whis=1.5, showcaps=True, showbox=True, showfliers=False
)

sns.stripplot(
    x="label", y="distance_km", data=df_rev,
    jitter=True, color="black", alpha=0.3, size=2, ax=ax
)

# Color the boxes red
for i, artist in enumerate(ax.patches):
    # Each box is a Rectangle object
    artist.set_edgecolor('red')
    artist.set_linewidth(2)
    artist.set_facecolor('none')  # Optional

plt.xticks(rotation=45)
plt.title(f"Boxplot of distance from reference (km) per label for {CHU_name} addresses")

plt.tight_layout()
plt.savefig(f'./figures/boxplot_distance_km_{CHU_name}.png')
plt.show()


In [ ]:
df_all = df_rev.copy()
df_all['distance_m'] = df_all['a'].astype(float)
df_all['diff_revenu'] = df_all['diff_revenu'].astype(float)
df_mean = df_all.groupby('label')[['distance_km','diff_revenu']].mean().apply(lambda x : np.round(x,3)).rename({'distance_km':'dist_moy_km','diff_revenu':'diff_moy_revenu'},axis=1)
df_med = df_all.groupby('label')[['distance_km','diff_revenu']].median().apply(lambda x : np.round(x,3)).rename({'distance_km':'dist_med_km','diff_revenu':'diff_med_revenu'},axis=1)
df_all = pd.concat([df_mean,df_med],axis=1)

In [ ]:
df_all = df_all.reset_index()
df_all = df_all.rename({'label':'biais'})

import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure(data=[
                 go.Table(
                    header=dict(values=list(df_all.columns),align='center'),
                    cells=dict(values=df_all.values.transpose(),
                               fill_color = [["white","lightgrey"]*df_all.shape[0]],
                               align='center'
                              )
                        )
                   ])
# fig.update_layout(
#     # autosize=False,
#     width=500, height=1000)
# fig.update_yaxes(automargin='bottom+top')

fig.show()
# fig.write_image('fig_impact_biais_ref.png',scale=6)